# Local Aya 8B + SafeTune + AuditKIT

Run this notebook from the cloned repository after the README setup. This is a six-prompt benign integration check, not a safety benchmark. It launches an isolated process and saves the actual run results. It does not upload artifacts. The underlying runner has been tested on MPS; this notebook is a wrapper.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, subprocess
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'configs/runtime.json').exists(), 'Open from the repository or notebooks directory.'
PYTHON = ROOT / '.venv/bin/python'
assert PYTHON.exists(), 'Run python3.12 scripts/bootstrap_local.py first.'
subprocess.run([str(PYTHON), 'scripts/bootstrap_local.py', '--verify-only'], cwd=ROOT, check=True)


In [ ]:
subprocess.run([str(PYTHON), 'scripts/cache_model.py', '--offline'], cwd=ROOT, check=True)
print(json.loads((ROOT / 'configs/runtime.json').read_text()))


In [ ]:
attempt = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = ROOT / 'results' / ('aya-auditkit-notebook-' + attempt)
subprocess.run([str(PYTHON), '-u', 'scripts/auditkit_smoke_check.py', '--output', str(OUTPUT)], cwd=ROOT, check=True)


In [ ]:
report = json.loads((OUTPUT / 'report.json').read_text())
assert report['status'] == 'passed', report
print(json.dumps(report, indent=2, ensure_ascii=False))
result = json.loads((OUTPUT / 'auditkit-result.json').read_text())
for row in result['predictions']:
    print(row['sample_id'], repr(row['raw_output']), 'expected:', repr(row['expected']))


Interpretation: an execution pass means six real outputs were generated and scored without errors. Low model scores would still be valid measurements. Do not use this tiny fixture as evidence of multilingual safety or as the actual held-out benchmark. Keep outputs and provenance for later HF publication.
